<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/08-operations/03-experiment-tracking-and-registry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment Tracking & Registry

**Goal:** Use MLflow end to end. Log an eval run's params/metrics/artifacts, compare runs to make a decision, then register and stage-promote the winning configuration. The tooling that turns "I ran an eval once" into a tracked, reproducible, promotable workflow.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Why this notebook exists

Section 04 built an eval harness: run a config, get a number. Section 08/01 traced production calls. But between "I got a number in a notebook" and "our team ships model changes safely" sits a gap: **where do the numbers live, how do you compare last week's run to today's, and which exact config is in production?**

That's experiment tracking + a model registry, and **MLflow** is the industry-standard open-source tool for it. This is the first notebook in the repo to reach for a real framework rather than raw APIs, and deliberately so, because tracking is *infrastructure*, not a pattern you'd hand-roll in production. (Contrast section 04, which taught the eval *logic* from scratch so you understand what's being tracked.)

> **⭐ Key takeaway —** MLflow doesn't run your evals — it *remembers* them. Every run's params, metrics, and artifacts become queryable history, and the registry makes "which version is live?" a lookup instead of tribal knowledge. That memory is what makes a model change a *decision* instead of a guess.

## Setup

Self-contained as always. Two differences from the rest of the repo:

1. This notebook installs **MLflow** (the tool it teaches), not just `aien`.
2. It needs **no API key**. To keep the focus on MLflow, the "system under test" is a small deterministic scorer (section 02's golden-set idea). The one cell where a real model call would go is marked, so you can see exactly where your section-04 harness plugs in.

In [ ]:
%pip install -q mlflow "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
import mlflow

# MLflow logs to a local ./mlruns directory by default — no server, no account needed.
# In a team you'd point tracking_uri at a shared MLflow server instead; the code is identical.
mlflow.set_experiment("rag-eval")   # groups related runs; created on first use
print("MLflow", mlflow.__version__, "-> logging to ./mlruns")

## The system under test (a stand-in for your section-04 harness)

To keep this runnable without a key, the "RAG system" is a deterministic scorer over a tiny golden set: each config produces a quality score and a cost. In your real project, **this function is your section-04 `run_eval`**. It would call the model, score against the golden set, and return the same shape. MLflow doesn't care which; it tracks whatever numbers you hand it.

In [ ]:
# A tiny golden set (section 02/04 idea): questions with a known-good answer key.
GOLDEN = [
    {"q": "What port does the protocol use?", "key": "8080"},
    {"q": "Who ratified the spec?", "key": "the working group"},
    {"q": "What is the max message size?", "key": "64 KB"},
]

def evaluate(config):
    """Stand-in for section 04's run_eval. Returns metrics MLflow will track.
    Real version: call the model with `config`, score outputs against GOLDEN."""
    # Deterministic toy scoring: more chunks + a reranker -> higher accuracy, higher cost.
    base = 0.55 + 0.08 * config["top_k"] + (0.12 if config["rerank"] else 0.0)
    accuracy = min(base, 0.98)
    cost_per_query = 0.0002 * config["top_k"] * (2.0 if config["rerank"] else 1.0)
    latency_ms = 120 + 40 * config["top_k"] + (200 if config["rerank"] else 0)
    return {"accuracy": round(accuracy, 3),
            "cost_per_query": round(cost_per_query, 5),
            "latency_ms": latency_ms}

print(evaluate({"top_k": 3, "rerank": False}))

## Log a run: params in, metrics + artifacts out

The core MLflow loop is one context manager: open a run, log the **params** (the config), log the **metrics** (the results), log any **artifacts** (files: the eval report, a plot, the golden set). Everything inside `with mlflow.start_run()` is captured together as one comparable record.

In [ ]:
import json, tempfile, os

def tracked_eval(config, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(config)                 # the knobs you turned
        metrics = evaluate(config)
        mlflow.log_metrics(metrics)               # the numbers you got

        # Log the full result as an artifact — the eval "report" travels with the run.
        report = {"config": config, "metrics": metrics, "golden_set_size": len(GOLDEN)}
        path = os.path.join(tempfile.mkdtemp(), "eval_report.json")
        with open(path, "w") as f:
            json.dump(report, f, indent=2)
        mlflow.log_artifact(path)
        return metrics, run.info.run_id

m, _ = tracked_eval({"top_k": 3, "rerank": False}, run_name="baseline")
print("logged baseline:", m)

## Sweep configs — this is what tracking is *for*

One run is a log line. The payoff is many runs you can compare. Below we sweep the RAG knobs, and each becomes a tracked run with its params and metrics side by side. This is exactly the "ship a change, measure it" loop from section 04, now with a memory.

In [ ]:
sweep = [
    {"top_k": 1, "rerank": False},
    {"top_k": 3, "rerank": False},
    {"top_k": 5, "rerank": False},
    {"top_k": 3, "rerank": True},
    {"top_k": 5, "rerank": True},
]
for cfg in sweep:
    name = f"k{cfg['top_k']}{'_rerank' if cfg['rerank'] else ''}"
    tracked_eval(cfg, run_name=name)
print(f"logged {len(sweep)} runs to the 'rag-eval' experiment")

## Compare runs and pick a winner, programmatically

MLflow has a UI (`mlflow ui` locally, or the tracking-server web app on a team), but everything is queryable in code too, which is how you'd gate a decision in CI. Pull the runs back and rank them by the trade-off you care about.

In [ ]:
runs = mlflow.search_runs(experiment_names=["rag-eval"])
# Show the columns that matter, best accuracy first.
cols = ["tags.mlflow.runName", "params.top_k", "params.rerank",
        "metrics.accuracy", "metrics.cost_per_query", "metrics.latency_ms"]
view = runs[cols].sort_values("metrics.accuracy", ascending=False)
print(view.to_string(index=False))

# A real gate is rarely "max accuracy" — it's the best accuracy under a cost/latency budget.
budget = runs[(runs["metrics.cost_per_query"] <= 0.002) & (runs["metrics.latency_ms"] <= 400)]
winner = budget.sort_values("metrics.accuracy", ascending=False).iloc[0]
print(f"\nwinner under budget: {winner['tags.mlflow.runName']} "
      f"(acc={winner['metrics.accuracy']}, ${winner['metrics.cost_per_query']}/q, "
      f"{int(winner['metrics.latency_ms'])}ms)")

> **🔵 Interview signal —** "we picked the config with the best accuracy *under our cost and latency budget*, and the run is logged so anyone can reproduce it" is a production-grade answer. It shows you optimize against constraints (09/02, 10/01) and that your choice is auditable — not "it felt better."

## Register the winner: the model registry

Tracking remembers *experiments*. The **registry** answers a different question: *which version is the one we bless for production?* You register a model as a named entry, and it gets a version number. Then you point a moving **alias** (like `champion`) at the blessed version, so "what's live?" is a lookup, and rolling back is re-pointing the alias at the previous version, not a scramble.

(MLflow 3 uses *aliases* for this; older tutorials use *stages* like `Staging`/`Production`, which are now deprecated. Same idea, newer API.)

To register something meaningful, we log the winning config as a tiny **pyfunc model** (a real, loadable artifact) rather than just a JSON file. In your project this is your actual RAG pipeline; the registry flow is identical either way.

In [ ]:
from mlflow import MlflowClient
from mlflow.pyfunc import PythonModel

MODEL_NAME = "rag-answerer"
client = MlflowClient()

# The winning config from the search above.
winner_cfg = {"top_k": int(winner["params.top_k"]), "rerank": winner["params.rerank"] == "True"}

class RagAnswerer(PythonModel):
    """Stand-in for your real RAG pipeline. In production predict() would retrieve + generate."""
    def predict(self, context, model_input):
        return [f"answer (top_k={winner_cfg['top_k']}, rerank={winner_cfg['rerank']})"
                for _ in range(len(model_input))]

# Log the model inside a run, then register that logged model as a named version.
with mlflow.start_run(run_name="register-winner"):
    mlflow.log_params(winner_cfg)
    mlflow.log_metrics(evaluate(winner_cfg))
    info = mlflow.pyfunc.log_model(name="model", python_model=RagAnswerer())

mv = mlflow.register_model(model_uri=info.model_uri, name=MODEL_NAME)
print(f"registered {MODEL_NAME} version {mv.version}")

# Point the 'champion' alias at this version. Re-pointing it later IS the promotion/rollback.
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f"alias champion -> {MODEL_NAME} v{mv.version}")

In [ ]:
# "What is live right now?" — a lookup by alias, not tribal knowledge.
live = client.get_model_version_by_alias(MODEL_NAME, "champion")
run = client.get_run(live.run_id)
print(f"CHAMPION: {MODEL_NAME} v{live.version}")
print(f"  config : top_k={run.data.params['top_k']}, rerank={run.data.params['rerank']}")
print(f"  metrics: accuracy={run.data.metrics['accuracy']}, "
      f"cost/q=${run.data.metrics['cost_per_query']}")
print("  -> rollback = re-point 'champion' at the previous version; audit = read this run's params")

> **⚠️ Production reality —** the registry is only as trustworthy as the discipline around it. Register *only* configs that passed the eval gate (section 04), and make promotion to `Production` a reviewed step, not an automatic one. A registry full of un-evaluated "latest" versions is just a fancier way to lose track — the tool doesn't create the rigor, it *records* it.

## Where this sits in the stack

MLflow closes the loop the repo has been building:

- **Section 02/04 evals** produce the numbers → **08/03 (here)** tracks and compares them.
- **08/01 observability** produces production metrics → feed drift signals back as new tracked runs.
- **10/01 system design** picks a config under budget → the registry records *which* version, so the deployed service and the eval are provably the same thing.

That traceability, from an eval number to the exact version serving traffic, is the "MLOps" an AI-systems team expects you to know exists, even if a platform team runs the server.

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Log params, metrics, *and* the eval report artifact together per run | Log a metric alone, and later you can't reproduce which config produced it |
| Pick the winner under a cost/latency budget, in code, so it's a gate | "This run looked best" by eyeballing the UI once |
| Register only configs that passed the section-04 eval gate | Register every "latest" run and lose the signal in the noise |
| Use an alias (e.g. `champion`) for what's live; rollback = re-point it | Track "what's live" in a Slack message or someone's memory |
| Point tracking_uri at a shared server for a team; code stays identical | Keep runs in a local ./mlruns nobody else can see, call it tracking |
| Treat the registry as a record of rigor you already have | Expect the tool to create discipline you didn't practice |

## Exercises

1. **Plug in the real harness.** Replace `evaluate()` with your actual section-04 `run_eval` (it needs a Groq key). Confirm the *entire* MLflow loop (log, sweep, compare, register) works unchanged. That invariance is the point: MLflow tracks whatever metrics you hand it.
2. **A stricter gate.** Change the budget filter to also require `latency_ms <= 300`. Note how the winner changes, register it as a new version, and point `champion` at it. Read the alias back and confirm it now resolves to the new version.
3. **See it in the UI.** Run `!mlflow ui` (locally) or read the `./mlruns` folder structure. Find where params, metrics, and your `eval_report.json` artifact are stored per run. Understanding the on-disk shape demystifies the tool.
4. **Rollback drill.** Register a *worse* config as a new version and point `champion` at it, then simulate a rollback by re-pointing `champion` back at the prior version. Write the two `set_registered_model_alias` calls and one sentence on why an alias makes this a 10-second operation instead of a redeploy.